In [2]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from pytorch_tcn import TCN

In [6]:
df = pd.read_pickle('../data/processed/df_train_val.pkl')
df['datetime'] = pd.to_datetime(df['datetime'], utc=True)
df = df.set_index('datetime')

In [13]:
def seq(X, y, seq_len, pred_len, stride):
    Xs, ys = [], []
    for i in range(0, len(X) - seq_len - pred_len + 1, stride):
        Xs.append(X[i:i+seq_len])
        ys.append(y[i+seq_len:i+seq_len+pred_len])
    return np.array(Xs), np.array(ys)

In [ ]:
t, _ = seq(
    X=df.drop(columns=['datetime', 'price']).to_numpy(), y=price,
    seq_len=24*7*4, pred_len=24, stride=24)
t.shape

In [3]:
class EPFDataset(torch.utils.data.Dataset):
    def __init__(self, X:torch.tensor, y:torch.tensor, seq_len:int, pred_len:int, stride:int):
        self.X = X
        self.y = y
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.stride = stride
    
    def __len__(self):
        return (len(self.X)-self.seq_len-self.pred_len)//self.stride + 1
    
    def __getitem__(self, i):
        num_strides = i*self.stride
        return (
            self.X[num_strides : num_strides+self.seq_len],
            self.y[num_strides+self.seq_len : num_strides+self.seq_len+self.pred_len]
        )

In [12]:
d = EPFDataset(
    X=torch.tensor(df.drop(columns='price').to_numpy()), y=torch.tensor(df['price'].to_numpy()),
    seq_len=24*7*4, pred_len=24, stride=24
)
d[0]

(tensor([[ 2.5000e-01,  7.0500e+00,  9.2500e+01,  ...,  3.0600e+01,
           3.1700e+01, -6.0500e-01],
         [ 1.0000e-01,  7.2500e+00,  9.1250e+01,  ...,  3.1700e+01,
           3.2200e+01, -1.9460e+00],
         [ 7.5000e-02,  7.2500e+00,  9.2250e+01,  ...,  3.3100e+01,
           3.3300e+01, -1.0170e+00],
         ...,
         [ 2.5000e-02,  3.1750e+00,  8.1750e+01,  ...,  4.1800e+01,
           3.7800e+01, -1.2096e+01],
         [ 7.5000e-02,  3.7000e+00,  7.9000e+01,  ...,  4.3700e+01,
           3.8700e+01, -1.2185e+01],
         [ 2.2500e-01,  3.7750e+00,  7.7500e+01,  ...,  4.4100e+01,
           3.8900e+01, -1.2088e+01]], dtype=torch.float64),
 tensor([48.0700, 47.4900, 44.5900, 45.9700, 46.4600, 55.8200, 67.6700, 72.8000,
         70.6300, 70.3800, 67.5100, 64.0300, 63.8400, 66.5600, 66.0000, 69.3300,
         71.4500, 72.8200, 69.8100, 59.5100, 55.7400, 53.8300, 49.3400, 46.5200],
        dtype=torch.float64))

In [16]:
epf_set = EPFDataset(
    X=torch.tensor(df.drop(columns='price').to_numpy()), y=torch.tensor(df['price'].to_numpy()),
    seq_len=24*7*4, pred_len=24, stride=24
)
epf_loader = DataLoader(epf_set, batch_size=1, shuffle=False, drop_last=True)

In [ ]:
class TCN_LSTM_MHA(nn.Module):

    def __init__(
            self,
            input_size:int,
            channel_sizes:list,
            kernel_size:int,
            hidden_sizes:list,
            tcn_dropout:float,
            lstm_dropouts:list,
            mha_dropout:float,
            mha_heads:int,
            output_size:int=24
        ):
        super().__init__()
        
        # TCN
        self.tcn = TCN(
            num_inputs=input_size,
            num_channels=channel_sizes,
            kernel_size=kernel_size,
            dilation_reset=16,
            dropout=tcn_dropout,
            use_skip_connections=True,
            input_shape='NLC'   # (batch_size, time_steps, feature_channels)
        )

        # LSTM
        self.lstm_dropouts = nn.ModuleList([nn.Dropout(_) for _ in lstm_dropouts])
        self.lstms = nn.ModuleList()
        for i in range(len(hidden_sizes)):
            self.lstms.append(nn.LSTM(
                input_size=channel_sizes[-1] if i==0 else hidden_sizes[i-1],
                hidden_size=hidden_sizes[i],
                batch_first=True
            ))
        
        # MHA
        self.mha = nn.MultiheadAttention(
            embed_dim=hidden_sizes[-1],
            num_heads=mha_heads,
            dropout=mha_dropout,
            batch_first=True
        )
        self.norm = nn.LayerNorm(hidden_sizes[-1])

        # head
        self.fc = nn.Linear(hidden_sizes[-1], output_size)

    def forward(self, x):

        # TCN
        x = self.tcn(x)
        
        # LSTM
        for lstm, do in zip(self.lstms, self.lstm_dropouts):
            lstm_out, _ = lstm(x)
            x = x+do(lstm_out)

        # MHA
        mha_out, _ = self.mha(x, x, x)
        x = self.norm(x+mha_out)

        # head
        return self.fc(x[:, -24:, :]).squeeze(-1)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'mps')